# Три полносвязные сети для определения тренда

**Задача:** по десяти последовательным ценам закрытия определить общий тренд следующих трёх периодов.

| Класс задания | Направление | Индекс внутри PyTorch |
|---:|---|---:|
| −1 | Нисходящий | 0 |
| 0 | Боковой | 1 |
| 1 | Восходящий | 2 |

Создаём три разные сети по образцу `task3.py`. Используем файл `rosneft_close.csv` со столбцами `NUMBER;CLOSE`. Файл должен находиться рядом с ноутбуком; в Colab его нужно загрузить в текущую папку.

Нейронная сеть — функция с настраиваемыми весами и смещениями. На обучении она получает десять прошлых цен, выдаёт три оценки классов и меняет параметры, уменьшая ошибку относительно известного ответа. Будущие цены используются только для подготовки правильных ответов и не поступают на вход сети.

Предсказываем один класс общего тренда, а не три будущие цены.

## 1. Библиотеки и параметры

`pandas` читает CSV и показывает таблицы, `torch` работает с тензорами и обучает сети, `nn` содержит слои и функцию ошибки.

`SIDE_LIMIT = 1.5` сохраняем из образца: это абсолютный порог наклона в единицах цены за период, а не фиксированный процент. Условие задания не задаёт порог бокового тренда.

`NUM_EPOCHS = 2000` — количество проходов обучения. `manual_seed` фиксирует генератор случайных чисел, чтобы начальные веса были воспроизводимыми при одинаковых условиях запуска.

In [1]:
import pandas as pd
import torch
from torch import nn

SIDE_LIMIT = 1.5  # Единицы цены за один период.
NUM_EPOCHS = 2000

torch.manual_seed(42)

## 2. Чтение цен

Указываем разделитель `;`, выбираем только `CLOSE` и превращаем столбец в список. Порядок записей сохраняется; он должен соответствовать последовательным периодам. `NUMBER` на вход сети не подаётся.

In [2]:
closing_prices = pd.read_csv("rosneft_close.csv", sep=";")["CLOSE"].tolist()

print("Количество цен:", len(closing_prices))
print("Первые десять цен:", closing_prices[:10])

Количество цен: 675
Первые десять цен: [527.4, 508.0, 516.9, 510.45, 517.25, 524.15, 531.85, 530.55, 532.25, 532.45]


## 3. Определение правильного класса

По трём будущим ценам подбираем прямую $p(t)=kt+b$ методом наименьших квадратов. Вычисляем её наклон:

$$
k=\frac{\sum_{i=1}^{3}(t_i-\bar t)(p_i-\bar p)}
{\sum_{i=1}^{3}(t_i-\bar t)^2}, \qquad t=[1,2,3].
$$

`sum` складывает значения, `zip(time_steps, values)` соединяет номер периода с его ценой. Числитель связывает изменения времени и цены, знаменатель учитывает расположение периодов.

- $k > 1.5$: восходящий тренд, класс `1`.
- $k < -1.5$: нисходящий тренд, класс `−1`.
- Иначе: боковой тренд, класс `0`.

Для трёх равноотстоящих точек формула упрощается до $k=(p_3-p_1)/2$. Поэтому правило определяет общее направление по наклону, а не требует роста или падения на каждом отдельном шаге.

In [3]:
def linear_slope(values):
    time_steps = [1, 2, 3]
    avg_time = sum(time_steps) / 3
    avg_value = sum(values) / 3

    cross_sum = sum(
        (time_step - avg_time) * (value - avg_value)
        for time_step, value in zip(time_steps, values)
    )
    time_sum = sum((time_step - avg_time) ** 2 for time_step in time_steps)
    return cross_sum / time_sum

def trend_label(forecast_prices):
    coefficient = linear_slope(forecast_prices)
    if coefficient > SIDE_LIMIT:
        return 1
    elif coefficient < -SIDE_LIMIT:
        return -1
    return 0

## 4. Формирование входов и ответов

Каждое окно содержит 13 цен: первые десять — вход, следующие три — для вычисления ответа.

| Начало окна | Вход | Цены для ответа |
|---:|---|---|
| 0 | Периоды 1–10 | Периоды 11–13 |
| 1 | Периоды 2–11 | Периоды 12–14 |
| 2 | Периоды 3–12 | Периоды 13–15 |

Из $N$ цен получается $N-13+1=N-12$ примеров. Правая граница среза Python не включается.

Из каждой входной цены вычитаем последнюю цену **этого же входного окна**, как в образце. Например, `[100, 102, 101]` превращается в `[-1, 1, 0]`. Это убирает абсолютный уровень цены, сохраняя движение внутри окна. Масштаб колебаний при этом сохраняется.

Прибавление `1` переводит классы `−1, 0, 1` в индексы `0, 1, 2`. `float32` используется для входных чисел, `long` — для целочисленных индексов классов. При $M$ примерах входной тензор имеет форму `(M, 10)`, а ответы — `(M,)`.

In [4]:
def prepare_samples(values):
    features = []
    labels = []

    for offset in range(len(values) - 12):
        past_prices = values[offset:offset + 10]
        next_prices = values[offset + 10:offset + 13]

        # Десять цен относительно последней цены входного окна.
        features.append([value - past_prices[-1] for value in past_prices])
        # Индексы классов: 0 — вниз, 1 — боковой, 2 — вверх.
        labels.append(trend_label(next_prices) + 1)

    return (
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long),
    )

## 5. Обучающая и тестовая выборки

Первые 80% цен используются для обучения, последние 20% — для проверки. Сначала разделяем ряд и только потом формируем окна: цены из тестового участка не используются в обучающих примерах.

Соседние окна внутри одной выборки могут пересекаться. Между обучением и тестом пересечений нет. Окна, пересекающие границу участков, не используем.

In [5]:
split_index = int(len(closing_prices) * 0.8)
X_train, y_train = prepare_samples(closing_prices[:split_index])
X_test, y_test = prepare_samples(closing_prices[split_index:])

print("Обучающие входы:", tuple(X_train.shape))
print("Обучающие ответы:", tuple(y_train.shape))
print("Тестовые входы:", tuple(X_test.shape))
print("Тестовые ответы:", tuple(y_test.shape))

Обучающие входы: (528, 10)
Обучающие ответы: (528,)
Тестовые входы: (123, 10)
Тестовые ответы: (123,)


## 6. Три разные сети

В полносвязном слое каждый нейрон получает все значения предыдущего слоя и вычисляет $z=\sum_j w_jx_j+b$. Веса $w_j$ и смещение $b$ меняются при обучении.

`nn.Linear(a, b)` преобразует `a` значений в `b`. `nn.Sequential` выполняет слои последовательно.

| Сеть | Размеры слоёв | Активация |
|---|---|---|
| 1 | 10 → 24 → 3 | Tanh |
| 2 | 10 → 32 → 3 | ReLU |
| 3 | 10 → 24 → 12 → 3 | Tanh |

`Tanh` преобразует числа в диапазон от −1 до 1. `ReLU(z)=max(0,z)` заменяет отрицательные числа нулями. Эти функции добавляют нелинейность: без них несколько линейных слоёв можно было бы объединить в один.

Скрытые размеры `24`, `32` и `12` — выбранное число нейронов, не число периодов. Последние три выхода всегда соответствуют индексам классов: `0` — вниз, `1` — боковой, `2` — вверх. Выходы являются оценками, а не вероятностями.

In [6]:
classifiers = {
    "Сеть 1: 10 → 24 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 3),
    ),
    "Сеть 2: 10 → 32 → 3, ReLU": nn.Sequential(
        nn.Linear(10, 32),
        nn.ReLU(),
        nn.Linear(32, 3),
    ),
    "Сеть 3: 10 → 24 → 12 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 12),
        nn.Tanh(),
        nn.Linear(12, 3),
    ),
}

## 7. Обучение

Каждая сеть обучается независимо на всей обучающей выборке, содержащей разные классы.

`CrossEntropyLoss` сравнивает три оценки сети с индексом правильного класса. Дополнительный `Softmax` перед ней не нужен: необходимое преобразование предусмотрено внутри функции ошибки. Для одного примера смысл ошибки — $-\ln p_{\text{правильного класса}}$.

`Adam` обновляет параметры, полученные через `classifier.parameters()`. `lr=0.01` задаёт скорость обучения.

В каждой эпохе:

1. `zero_grad()` очищает градиенты предыдущего шага.
2. `classifier(X_train)` вычисляет оценки всех обучающих примеров, получая тензор `(M, 3)`.
3. Функция ошибки сравнивает оценки с правильными классами.
4. `backward()` вычисляет градиенты: как параметры влияют на ошибку. Параметры пока не меняются.
5. `step()` изменяет веса и смещения.

`train()` включает режим обучения. Словарь `final_errors` сохраняет последнюю ошибку каждой сети для последующего вывода.

In [7]:
final_errors = {}

for title, classifier in classifiers.items():
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)

    classifier.train()
    for iteration in range(NUM_EPOCHS):
        optimizer.zero_grad()
        logits = classifier(X_train)
        error = criterion(logits, y_train)
        error.backward()
        optimizer.step()

    final_errors[title] = error.item()

## 8. Проверка точности

`eval()` включает режим проверки, `no_grad()` отключает вычисление градиентов. Параметры больше не обновляются.

`argmax(dim=1)` выбирает индекс максимальной оценки в каждой строке. Например, `[2.4, -0.7, 0.8]` даёт индекс `0`, то есть нисходящий тренд.

Сравнение с правильными классами возвращает `True` и `False`. `.float()` преобразует их в `1.0` и `0.0`, `.mean()` вычисляет долю верных ответов, `.item()` извлекает обычное число Python. При форматировании результатов переводим долю в проценты.

In [8]:
def compute_accuracy(classifier, features, labels):
    classifier.eval()
    with torch.no_grad():
        predicted_labels = classifier(features).argmax(dim=1)
    return (predicted_labels == labels).float().mean().item()

evaluation = []
for title, classifier in classifiers.items():
    evaluation.append({
        "Сеть": title,
        "Ошибка обучения": final_errors[title],
        "Точность на обучении, %": compute_accuracy(classifier, X_train, y_train) * 100,
        "Точность на тесте, %": compute_accuracy(classifier, X_test, y_test) * 100,
    })

pd.DataFrame(evaluation).round(2)

,Сеть,Ошибка обучения,"Точность на обучении, %","Точность на тесте, %"
0,"Сеть 1: 10 → 24 → 3, Tanh",0.47,79.92,29.27
1,"Сеть 2: 10 → 32 → 3, ReLU",0.45,82.95,31.71
2,"Сеть 3: 10 → 24 → 12 → 3, Tanh",0.16,93.37,25.20


Высокая точность на обучении при низкой точности на тесте указывает на переобучение: сеть хорошо подстроилась под обучающие примеры, но плохо переносит это на более поздний участок ряда. Код реализует задачу классификации, однако качество прогноза нужно оценивать по тестовым результатам.